# Community Fish Detector × FiftyOne — Demo Notebook

A hands-on tour of [FiftyOne](https://docs.voxel51.com) built on the
[Community Fish Detector (CFD)](https://github.com/filippovarini/community-fish-detector)
and the [Community Fish Detection Dataset](https://lila.science/datasets/community-fish-detection-dataset/).

**The story:** a single `fish` class trained on ~17 wildly different source datasets —
murky brackish water, tropical reef GoPro footage, freshwater underwater-video billabongs,
and 32 cm lab zebrafish tanks. The dataset preserves a per-image `dataset` provenance
field, so we run the model, evaluate it, and then **slice performance by source
environment** to expose exactly where a general-purpose detector breaks down.

**What this notebook does**
- Streams a small, curated, license-safe subset of the dataset (no bulk download) and loads
  it into FiftyOne with ground-truth boxes and per-image source provenance.
- Runs the RF-DETR CFD model, evaluates it, and slices performance by source environment.
- Computes embeddings, similarity, uniqueness, and label-mistake signals.
- Persists **every demo moment as a named _saved view_.** Saved views store the view
  *definition* (filters, sorts, limits), not copies of the data, so they re-evaluate live
  against the dataset. Recreating any demo state is one click in the App's view-selector
  dropdown (the bookmark icon, top-left), or one `dataset.load_saved_view(name)` call.
- Tolerates sources whose annotations are polygon segmentations (no `bbox`) or RLE masks.

> Defaults touch only CC-BY / CC0 / CDLA / Apache / MIT sources, so the curated subset is
> safe to show publicly. NC-restricted sources are skipped.


## 0. Prerequisites & environment

Works on macOS, Linux, and Windows. FiftyOne and RF-DETR both support Python 3.10–3.12.
Create and activate a fresh virtual environment, then launch Jupyter from inside it so this
notebook's kernel is that environment:

```bash
python3 -m venv .venv                 # Python 3.10-3.12
source .venv/bin/activate             # macOS / Linux
# .venv\Scripts\activate             # Windows (PowerShell/cmd)
pip install jupyter
jupyter lab                           # or: jupyter notebook
```

The install cell below upgrades FiftyOne and installs the model + helper packages **into
the running kernel**. RF-DETR pulls a PyTorch wheel that auto-selects CPU / CUDA / Apple MPS.

> Using a fresh environment avoids dependency clashes. If pip prints a conflict warning about
> an unrelated package already installed elsewhere, it's generally safe to ignore for this
> demo. **Restart the kernel after the first install** so the new packages load.


In [ ]:
# Installs into THIS kernel's environment. Safe to re-run. Restart the kernel afterward.
%pip install --upgrade fiftyone
%pip install --upgrade "rfdetr" "supervision" "umap-learn" "requests" "pillow"

import fiftyone as fo
print("FiftyOne:", fo.__version__)

## 1. Configuration

Everything tunable lives here. `TARGET_SOURCES` is a *wish list* matched (case-insensitive
substring) against the real `dataset` values in the metadata, so a rename or typo degrades
gracefully. We pick a spread of environments on purpose, to make the domain gap visible.


In [ ]:
from pathlib import Path

WORK_DIR    = Path("./cfd_fiftyone_demo").resolve()
IMAGES_DIR  = WORK_DIR / "images"
META_DIR    = WORK_DIR / "metadata"
WEIGHTS_DIR = WORK_DIR / "weights"
for d in (IMAGES_DIR, META_DIR, WEIGHTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Wish list spanning very different visual domains (matched by substring).
TARGET_SOURCES = [
    "coralscapes",     # tropical reef, diver GoPro (Apache-2.0)
    "roboflow",        # web-sourced fish photos (CC0)
    "deepfish",        # Australian seafloor habitats (MIT) - segmentation-derived boxes
    "brackish",        # murky brackish water, Denmark (CC-BY-SA)
    "zebrafish",       # lab tank, top-down (CC-BY-4.0)
    "mit_sea_grant",   # river herring, freshwater video frames (CDLA)
    "puget",           # nearshore aquaculture video frames (CDLA)
]

IMAGES_PER_SOURCE = 60      # cap per source; lower for a faster run
ONLY_VAL_SPLIT    = True    # use the reference validation split only
PREFER_BOXED      = True    # bias sampling toward images that contain fish

MODEL_CHOICE   = "nano"     # one of: "nano", "small", "medium"
CONF_THRESHOLD = 0.25       # lower than repo's 0.3 to surface more false positives

DATASET_NAME = "community-fish-detector-demo"

LILA_BASE    = "https://lilawildlife.blob.core.windows.net/lila-wildlife/community-fish-detection-dataset"
META_ZIP_URL = f"{LILA_BASE}/community_fish_detection_dataset.json.zip"

MODEL_WEIGHTS = {
    "nano":   ("https://github.com/filippovarini/community-fish-detector/releases/download/"
               "cfd-2026.02.02-rf-detr-nano/community-fish-detector-2026.02.02-rf-detr-nano-640.pth", 640),
    "small":  ("https://github.com/filippovarini/community-fish-detector/releases/download/"
               "2026.05.13-release/fish-detector-rf-detr-small-1024-2026.06.06-checkpoint_16.stripped.pth", 1024),
    "medium": ("https://github.com/filippovarini/community-fish-detector/releases/download/"
               "2026.05.13-release/fish-detector-rf-detr-medium-1024-2026.03.24-checkpoint_11.stripped.pth", 1024),
}
print("Workspace:", WORK_DIR)

## 2. Download & parse the COCO metadata

The full dataset is >1.9M images / >935k boxes, but the **metadata** is a single COCO JSON.
We download the zip once and load it.

> **Memory note:** the unzipped JSON loads to ~2–4 GB in RAM — fine on most machines. On a
> low-memory machine, swap `json.load` for a streaming parser (`ijson`); nothing else changes.

Beyond standard COCO fields, each image carries `dataset` (source), `is_train` (split), and
`original_data_source`.


In [ ]:
import json, zipfile, requests

def download(url, dest: Path, desc=""):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"✓ cached: {dest.name}")
        return dest
    print(f"↓ downloading {desc or dest.name} ...")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total, done = int(r.headers.get("content-length", 0)), 0
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk); done += len(chunk)
                if total:
                    print(f"\r  {done/1e6:7.1f} / {total/1e6:7.1f} MB", end="")
        print()
    return dest

meta_zip = download(META_ZIP_URL, META_DIR / "cfd_metadata.json.zip", desc="COCO metadata")
with zipfile.ZipFile(meta_zip) as z:
    json_name = next(n for n in z.namelist() if n.endswith(".json"))
    with z.open(json_name) as f:
        coco = json.load(f)

print("images:", len(coco["images"]), "| annotations:", len(coco["annotations"]),
      "| categories:", [c["name"] for c in coco["categories"]])

In [ ]:
from collections import defaultdict

images_by_id = {img["id"]: img for img in coco["images"]}
anns_by_image = defaultdict(list)
for a in coco["annotations"]:
    anns_by_image[a["image_id"]].append(a)

by_source = defaultdict(list)
for img in coco["images"]:
    by_source[img.get("dataset", "unknown")].append(img)

print(f"{'source':<40}{'total':>10}{'val':>10}{'boxed':>10}")
print("-" * 70)
for src in sorted(by_source, key=lambda s: -len(by_source[s])):
    imgs = by_source[src]
    n_val = sum(1 for i in imgs if not i.get("is_train", True))
    n_box = sum(1 for i in imgs if anns_by_image.get(i["id"]))
    print(f"{src:<40}{len(imgs):>10}{n_val:>10}{n_box:>10}")

## 3. Curate a small, diverse subset

Match the wish list to real source names, sample up to `IMAGES_PER_SOURCE` validation
images per source (preferring images that contain fish), and download just those.


In [ ]:
import random
random.seed(51)

real_sources = list(by_source.keys())

def resolve(wish):
    w = wish.lower()
    return [s for s in real_sources if w in s.lower()]

chosen_sources = []
for wish in TARGET_SOURCES:
    for s in resolve(wish):
        if s not in chosen_sources:
            chosen_sources.append(s)
if not chosen_sources:
    print("No wish-list sources matched; using the largest available sources.")
    chosen_sources = sorted(by_source, key=lambda s: -len(by_source[s]))[:5]
print("Selected sources:", chosen_sources)

selected_images = []
for src in chosen_sources:
    pool = by_source[src]
    if ONLY_VAL_SPLIT:
        val = [i for i in pool if not i.get("is_train", True)]
        pool = val or pool
    if PREFER_BOXED:
        boxed   = [i for i in pool if anns_by_image.get(i["id"])]
        unboxed = [i for i in pool if not anns_by_image.get(i["id"])]
        random.shuffle(boxed); random.shuffle(unboxed)
        pool = boxed + unboxed
    else:
        random.shuffle(pool)
    take = pool[:IMAGES_PER_SOURCE]
    selected_images.extend(take)
    print(f"  {src:<40} sampled {len(take)}")
print("Total images to fetch:", len(selected_images))

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_image(img):
    file_name = img["file_name"]
    local = IMAGES_DIR / file_name
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.exists() and local.stat().st_size > 0:
        return img["id"], local, None
    url = f"{LILA_BASE}/{file_name}"
    try:
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(local, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 16):
                    f.write(chunk)
        return img["id"], local, None
    except Exception as e:
        return img["id"], None, str(e)

local_paths, errors = {}, []
with ThreadPoolExecutor(max_workers=16) as ex:
    futures = [ex.submit(fetch_image, img) for img in selected_images]
    for i, fut in enumerate(as_completed(futures), 1):
        iid, path, err = fut.result()
        (errors.append((iid, err)) if err else local_paths.__setitem__(iid, path))
        if i % 25 == 0 or i == len(futures):
            print(f"\r  fetched {i}/{len(futures)} (errors: {len(errors)})", end="")
print()
if errors:
    print("Some images failed and will be skipped:", errors[:3], "...")
print("Images available locally:", len(local_paths))

### Build the FiftyOne dataset (with a robust box converter)

COCO boxes are `[x, y, w, h]` absolute pixels; FiftyOne wants them normalized to `[0, 1]`.
Some CFD sources (e.g. DeepFish, F4K) came from **segmentation masks**, so a subset of
annotations have no `bbox`. The converter below:

- uses `bbox` when present,
- derives a tight box from a **polygon** `segmentation` when `bbox` is missing,
- skips **RLE** masks (dict-encoded) and degenerate boxes,

and reports how many annotations it skipped, so you can spot a source that's mask-only.


In [ ]:
from PIL import Image

if DATASET_NAME in fo.list_datasets():
    fo.delete_dataset(DATASET_NAME)
dataset = fo.Dataset(DATASET_NAME, persistent=True)

skipped_anns = 0

def _bbox_from_polygon(seg):
    """Tight [x, y, w, h] (abs px) from a COCO polygon segmentation; None if unusable."""
    if not isinstance(seg, list) or not seg:
        return None  # RLE dict or empty -> skip
    xs, ys = [], []
    for poly in seg:
        if not isinstance(poly, (list, tuple)):
            return None
        xs.extend(poly[0::2]); ys.extend(poly[1::2])
    if not xs or not ys:
        return None
    return [min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)]

def coco_to_fo_boxes(img_record, W, H):
    global skipped_anns
    dets = []
    for a in anns_by_image.get(img_record["id"], []):
        bbox = a.get("bbox") or _bbox_from_polygon(a.get("segmentation"))
        if not bbox or len(bbox) != 4:
            skipped_anns += 1; continue
        x, y, w, h = bbox
        if w <= 0 or h <= 0:
            skipped_anns += 1; continue
        dets.append(fo.Detection(label="fish",
                                 bounding_box=[x / W, y / H, w / W, h / H]))
    return fo.Detections(detections=dets)

samples = []
for img in selected_images:
    path = local_paths.get(img["id"])
    if path is None:
        continue
    try:
        with Image.open(path) as im:
            W, H = im.size
    except Exception:
        W, H = img.get("width"), img.get("height")
        if not (W and H):
            continue
    s = fo.Sample(filepath=str(path))
    s["ground_truth"] = coco_to_fo_boxes(img, W, H)
    s["source"] = img.get("dataset", "unknown")
    s["split"]  = "train" if img.get("is_train", True) else "val"
    s["original_data_source"] = img.get("original_data_source")
    s["n_gt"] = len(s["ground_truth"].detections)
    samples.append(s)

dataset.add_samples(samples)
dataset.compute_metadata()
print(f"Skipped {skipped_anns} annotations without a usable box (RLE / degenerate).")
print(dataset)
print("\nImages per source:", dataset.count_values("source"))

## 4. First look in the App

Color by `source` and filter `ground_truth` to feel how different these environments look,
even though every box is just "fish".


In [ ]:
session = fo.launch_app(dataset)
session

## 5. Run the Community Fish Detector

Download the RF-DETR weights and run inference into a `predictions` field. Several warnings
are expected and harmless: the "not optimized for inference" notice only matters for GPU FP16
tensor cores, and the DINOv2 backbone / class-count messages are normal when loading a
fully-trained single-class checkpoint.


In [ ]:
weights_url, resolution = MODEL_WEIGHTS[MODEL_CHOICE]
weights_path = download(weights_url, WEIGHTS_DIR / Path(weights_url).name, desc=f"{MODEL_CHOICE} weights")
print("Weights:", weights_path, "| resolution:", resolution)

In [ ]:
from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium

_MODEL_CLASSES = {"nano": RFDETRNano, "small": RFDETRSmall, "medium": RFDETRMedium}
model = _MODEL_CLASSES[MODEL_CHOICE](pretrain_weights=str(weights_path), resolution=resolution)
print("Loaded RF-DETR", MODEL_CHOICE)

In [ ]:
def sv_to_fo(det, W, H, label="fish"):
    out = []
    xyxy = det.xyxy
    confs = det.confidence if det.confidence is not None else [None] * len(xyxy)
    for (x1, y1, x2, y2), c in zip(xyxy, confs):
        out.append(fo.Detection(
            label=label,
            bounding_box=[float(x1)/W, float(y1)/H, float(x2-x1)/W, float(y2-y1)/H],
            confidence=None if c is None else float(c),
        ))
    return fo.Detections(detections=out)

with fo.ProgressBar() as pb:
    for sample in pb(dataset):
        image = Image.open(sample.filepath).convert("RGB")
        W, H = image.size
        det = model.predict(image, threshold=CONF_THRESHOLD)
        sample["predictions"] = sv_to_fo(det, W, H)
        sample["n_pred"] = len(sample["predictions"].detections)
        sample.save()

print("Predictions per image (min, max):", dataset.bounds("n_pred"))

## 6. Evaluate overall (COCO mAP)

`evaluate_detections` tags each prediction `tp`/`fp` and each missed GT box `fn` in an
`eval` field, adds sample-level `eval_tp`/`eval_fp`/`eval_fn` counts, and computes COCO mAP.
These fields are what several saved views below filter on.


In [ ]:
results = dataset.evaluate_detections(
    "predictions", gt_field="ground_truth", eval_key="eval", compute_mAP=True,
)
print(f"Overall COCO mAP: {results.mAP():.3f}\n")
results.print_report()

## 7. The domain-gap reveal: performance **per source**

The payoff. A single number hides everything; per-source mAP shows where the detector is
production-ready and where it collapses. We capture the best/worst sources to save as views.


In [ ]:
import pandas as pd
from fiftyone import ViewField as F

rows = []
for src in dataset.distinct("source"):
    sview = dataset.match(F("source") == src)
    r = sview.evaluate_detections("predictions", gt_field="ground_truth",
                                  eval_key=None, compute_mAP=True)
    rows.append({"source": src, "images": sview.count(),
                 "gt_boxes": sview.sum("n_gt"), "pred_boxes": sview.sum("n_pred"),
                 "mAP": round(r.mAP(), 3)})

df = pd.DataFrame(rows).sort_values("mAP", ascending=False).reset_index(drop=True)
best_source  = df.iloc[0]["source"]
worst_source = df.iloc[-1]["source"]
print("best:", best_source, "| worst:", worst_source)
df

## 8. Embeddings, similarity, uniqueness, mistakenness

Compute the Brain runs that the analytical saved views depend on. In the App's
**Embeddings** panel, color by `source` — environments separate into clusters, the visual
explanation for the per-source mAP spread.


In [ ]:
import fiftyone.brain as fob

fob.compute_visualization(dataset, model="clip-vit-base32-torch",
                          brain_key="img_viz", embeddings="clip_embeddings")
fob.compute_similarity(dataset, embeddings="clip_embeddings", brain_key="img_sim")
fob.compute_uniqueness(dataset)
fob.compute_mistakenness(dataset, "predictions", label_field="ground_truth")

session.refresh()
print("Brain runs:", dataset.list_brain_runs())

## 9. Persist every demo moment as a saved view

This is the centerpiece. Each view below is stored on the (persistent) dataset by name, so:

- In the **App**, pick it from the view-selector dropdown (bookmark icon, top-left).
- In **code**, `dataset.load_saved_view(name)` or `session.view = dataset.load_saved_view(name)`.

Saved views store the *pipeline*, not the samples, so they always reflect the current data.
The helper is idempotent — re-running replaces existing views of the same name.

The last group uses `to_evaluation_patches`, which turns the view into **one crop per
true positive / false positive / false negative**. Those are the "see the failure" views:
open `eval-patches: false positives` for a wall of hallucinated fish, or
`eval-patches: missed fish` for everything the model walked past — far more legible on a
projector than error boxes buried in full scenes.

> **Single-class note.** `compute_mistakenness` leans on class *disagreement*, which barely
> exists with one class, and its `possible_missing` flag is gated at 0.95 confidence — higher
> than detection models usually reach — so those fields are often empty here. The
> missing-label views below therefore derive "confident fish, no GT box" straight from the
> evaluation's false positives at a reachable `MISSING_CONF` threshold, which actually fires.


In [ ]:
from fiftyone import ViewField as F

def save(name, view, description=""):
    if dataset.has_saved_view(name):
        dataset.delete_saved_view(name)
    dataset.save_view(name, view, description=description)
    print(f"  saved: {name:<34} ({view.count()} samples)")

# compute_mistakenness flags `possible_missing` only for FPs above a hard-coded 0.95
# confidence, which detection models rarely reach (so that field is usually empty).
# We derive the "confident fish, no GT box" signal from the evaluation instead, at a
# reachable threshold. Lower this if your model's confidences run low.
MISSING_CONF = 0.7

print("Per-source views:")
for src in dataset.distinct("source"):
    save(f"source: {src}", dataset.match(F("source") == src),
         description=f"All images from the '{src}' source dataset.")

print("\nDomain-gap views:")
save("best source (highest mAP)",
     dataset.match(F("source") == best_source),
     description=f"Highest-scoring source ({best_source}). Model does well here.")
save("worst source (lowest mAP)",
     dataset.match(F("source") == worst_source),
     description=f"Lowest-scoring source ({worst_source}). Where the detector struggles.")

print("\nError-analysis views:")
save("high-confidence false positives",
     dataset.filter_labels("predictions", (F("eval") == "fp") & (F("confidence") > 0.5))
            .sort_by(F("predictions.detections").length(), reverse=True),
     description="Confident detections that don't match any GT box (>0.5 conf).")
save("missed fish (false negatives)",
     dataset.filter_labels("ground_truth", F("eval") == "fn"),
     description="Ground-truth fish the model failed to detect.")
save("no detections",
     dataset.match(F("n_pred") == 0),
     description="Images where the model predicted nothing — candidate misses.")
save("crowded scenes",
     dataset.sort_by("n_pred", reverse=True).limit(25),
     description="Most detections per image — where false positives tend to hide.")

print("\nData-quality views:")
save("likely label mistakes",
     dataset.sort_by("mistakenness", reverse=True).limit(25),
     description="Highest mistakenness — probable GT annotation errors.")
save("possible missing annotations",
     dataset.filter_labels("predictions", (F("eval") == "fp") & (F("confidence") >= MISSING_CONF)),
     description="Confident (≥%.2f) fish predictions with no matching GT box — possible missing labels." % MISSING_CONF)
save("near-duplicate frames",
     dataset.sort_by("uniqueness").limit(25),
     description="Least-unique images — typically near-duplicate video frames.")
save("most unique / hardest",
     dataset.sort_by("uniqueness", reverse=True).limit(25),
     description="Most-unique images — rare or visually distinctive samples.")

print("\nEvaluation-patches views (one tile per error — the 'see the failure' payoff):")
try:
    eval_patches = dataset.to_evaluation_patches("eval")
    save("eval-patches: false positives",
         eval_patches.match(F("type") == "fp"),
         description="One crop per confident hallucinated fish (false positive).")
    save("eval-patches: missed fish",
         eval_patches.match(F("type") == "fn"),
         description="One crop per fish the model missed (false negative).")
    save("eval-patches: true positives",
         eval_patches.match(F("type") == "tp"),
         description="One crop per correct detection (true positive).")
except Exception as e:
    print("  (skipped patches views — not saveable on this build:", e, ")")

print("\nLabel-mistake views (spurious side — the 'missing' side is 'eval-patches: missed fish'):")
_spurious = dataset.match(F("possible_spurious") > 0).sort_by("possible_spurious", reverse=True)
if _spurious.count() > 0:
    save("possible spurious labels", _spurious,
         description="Images with GT boxes the model thinks shouldn't exist — possible over-labeling.")
else:
    print("  (no possible_spurious flags on this single-class run — skipping; use 'missed fish (false negatives)' instead)")
try:
    gt_patches = dataset.to_patches("ground_truth")
    save("label mistakes (patch view)",
         gt_patches.sort_by("ground_truth.mistakenness", reverse=True).limit(50),
         description="Top 50 ground-truth boxes ranked by mistakenness — one crop each.")
except Exception as e:
    print("  (skipped label-mistake patch view — not saveable on this build:", e, ")")

# Which sources have the most confident false positives (likely-missing labels)? Target those.
fp_conf = dataset.filter_labels("predictions", (F("eval") == "fp") & (F("confidence") >= MISSING_CONF))
missing_by_src = {s: fp_conf.match(F("source") == s).count("predictions.detections")
                  for s in dataset.distinct("source")}
top_missing_sources = [s for s, n in sorted(missing_by_src.items(), key=lambda kv: -kv[1]) if n > 0][:3]
try:
    missing_patches = (dataset.to_patches("predictions")
                       .match((F("predictions.eval") == "fp") & (F("predictions.confidence") >= MISSING_CONF))
                       .sort_by("predictions.confidence", reverse=True))
    if top_missing_sources:
        missing_patches = missing_patches.match(F("source").is_in(top_missing_sources))
    if missing_patches.count() > 0:
        save("missing labels (high-confidence)", missing_patches,
             description=("One crop per confident (≥%.2f) false positive — likely-missing labels" % MISSING_CONF)
                         + ((", worst sources: " + ", ".join(top_missing_sources)) if top_missing_sources else "")
                         + ". Review before filing upstream.")
    else:
        print("  (no confident FPs — lower MISSING_CONF to populate the missing-labels view)")
except Exception as e:
    print("  (skipped high-confidence missing-labels view:", e, ")")

print("\nAll saved views:")
for v in dataset.list_saved_views():
    print("  •", v)

## 10. Tour the saved views

Load any view by name. In a live demo you'd instead just pick these from the App dropdown.


In [ ]:
# Jump straight to the model's most embarrassing mistakes:
session.view = dataset.load_saved_view("high-confidence false positives")
print("Now showing:", "high-confidence false positives")

In [ ]:
# The domain-gap contrast, back to back:
session.view = dataset.load_saved_view("worst source (lowest mAP)")
print("Worst source in view. Swap to best with the line below.")
# session.view = dataset.load_saved_view("best source (highest mAP)")

In [ ]:
# Data-quality pass — probable label errors. Patch view = one crop per suspect box:
session.view = dataset.load_saved_view("label mistakes (patch view)")
print("Now showing one crop per suspect box, most-likely-wrong first.")
# Image-level version:  dataset.load_saved_view("likely label mistakes")
# Over-labeling side:    dataset.load_saved_view("possible spurious labels")

In [ ]:
# Missing-label bugs to file upstream: confident fish with no GT box, worst sources first.
_name = "missing labels (high-confidence)"
if dataset.has_saved_view(_name):
    session.view = dataset.load_saved_view(_name)
    print("Showing likely-missing labels in the most-affected sources.")
else:
    print("No missing-label view (no confident FPs — lower MISSING_CONF in the save cell).")

In [ ]:
# The "see the failure" payoff: a wall of individual error crops.
# Swap "false positives" <-> "missed fish" to flip between hallucinations and misses.
session.view = dataset.load_saved_view("eval-patches: false positives")
print("Now showing one crop per false positive. Try 'eval-patches: missed fish' too.")

## 11. Wrap-up & reuse

Everything persists with the dataset. In a fresh session you can skip straight to the
interesting parts:

```python
import fiftyone as fo
dataset = fo.load_dataset("community-fish-detector-demo")
session = fo.launch_app(dataset)
session.view = dataset.load_saved_view("worst source (lowest mAP)")
```

**Talking points**
- One `fish` class, many environments — the setup for a domain-gap story.
- The per-source mAP table turns one opaque number into a map of where the detector is
  production-ready and where it isn't.
- Embeddings visually explain that spread; the error and data-quality views close the loop
  from "the metric is low" to "here are the exact images to fix or relabel."
- Saved views make all of the above a one-click recreate during a live walkthrough.

**Extending**
- Swap `MODEL_CHOICE="medium"` (1024px) and compare per-source mAP to nano.
- Raise `IMAGES_PER_SOURCE` / add sources for a fuller picture.
- Add the video sources (Salmon CV, FishCLEF, F4K) as FiftyOne video datasets (some are NC).


In [ ]:
# Manage saved views / clean up.
print("Saved views:", dataset.list_saved_views())

# Delete one:        dataset.delete_saved_view("crowded scenes")
# Delete all views:  dataset.delete_saved_views()
# Delete dataset:    fo.delete_dataset(DATASET_NAME)